In [ ]:
!pip uninstall -y transformers accelerate
!pip install -U transformers datasets accelerate rouge_score sentencepiece

Found existing installation: transformers 5.6.2
Uninstalling transformers-5.6.2:
  Successfully uninstalled transformers-5.6.2
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Using cached transformers-5.6.2-py3-none-any.whl.metadata (33 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.6.2-py3-none-any.whl (10.4 MB)
Using cached accelerate-1.13.0-py3-none-any.whl (383 kB)


In [ ]:
import torch
import numpy as np

from datasets import load_dataset
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

from rouge_score import rouge_scorer

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

train_data = dataset["train"].select(range(5000))
val_data   = dataset["validation"].select(range(500))
test_data  = dataset["test"].select(range(500))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
model_name = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

In [ ]:
MAX_INPUT = 768
MAX_TARGET = 128

In [ ]:
def preprocess(example):

    inputs = tokenizer(
        example["article"],
        max_length=MAX_INPUT,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["highlights"],
        max_length=MAX_TARGET,
        truncation=True,
        padding="max_length"
    )

    inputs["labels"] = labels["input_ids"]

    return inputs

In [ ]:
tokenized_train = train_data.map(preprocess, batched=False)
tokenized_val   = val_data.map(preprocess, batched=False)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:
training_args = Seq2SeqTrainingArguments(

    output_dir="./bart_model",

    num_train_epochs=2,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-5,

    fp16=True,

    logging_steps=100,

    save_strategy="epoch",
    eval_strategy="epoch",

    predict_with_generate=True,

    report_to="none"
)

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    # tokenizer=tokenizer,
    data_collator=data_collator
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.326366,0.622052
2,1.728415,0.653561


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=2.39204306640625, metrics={'train_runtime': 1667.6998, 'train_samples_per_second': 5.996, 'train_steps_per_second': 0.75, 'total_flos': 1.625328451584e+16, 'train_loss': 2.39204306640625, 'epoch': 2.0})

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

trainer.save_model("/content/drive/MyDrive/bart_final")

Mounted at /content/drive


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

evaluating BART again

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/"))

['Dosti', 'Colab Notebooks', 'NAINITAL', 'IMG_20241026_195759.jpg', 'IMG_20241026_195618.jpg', 'Shivam_resume_dump.pdf', 'Screenshot 2025-07-22 at 11.02.37\u202fPM.png', 'Screenshot 2025-07-22 at 11.04.28\u202fPM.png', 'WhatsApp Image 2025-07-23 at 15.58.45.jpeg', 'WhatsApp Image 2025-07-23 at 15.57.17.jpeg', 'JHi5sj8OaFqtsmjW8lRfm2Jj9pMkVK7iLNMmKoHy7gNC_copy.pdf', 'Amazon-ML-Challenge', 'Shivam_Resume.pdf', 'Untitled document.gdoc', 'bart_final']


In [ ]:
model_path = "/content/drive/MyDrive/bart_final"

tokenizer = BartTokenizer.from_pretrained(model_path)
model = BartForConditionalGeneration.from_pretrained(model_path)

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

In [ ]:
def summarize(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT
    ).to(model.device)

    ids = model.generate(
        inputs["input_ids"],
        max_length=128,
        no_repeat_ngram_size=3,
        length_penalty=2.0,
        min_length=30,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(ids[0], skip_special_tokens=True)

In [ ]:
article = test_data[0]["article"]

print(summarize(article))
print("\nREFERENCE:\n")
print(test_data[0]["highlights"])

Palestinian Authority becomes 123rd member of the International Criminal Court .
The move gives the court jurisdiction over alleged crimes in Palestinian territories .
Israel and the United States opposed the Palestinians' efforts to join the body .

REFERENCE:

Membership gives the ICC jurisdiction over alleged crimes committed in Palestinian territories since last June .
Israel and the United States opposed the move, which could open the door to war crimes investigations against Israelis .


In [ ]:
scorer = rouge_scorer.RougeScorer(
    ['rouge1','rouge2','rougeL'],
    use_stemmer=True
)

r1,r2,rl = [],[],[]

for i in range(200):

    pred = summarize(test_data[i]["article"])
    ref  = test_data[i]["highlights"]

    score = scorer.score(ref, pred)

    r1.append(score["rouge1"].fmeasure)
    r2.append(score["rouge2"].fmeasure)
    rl.append(score["rougeL"].fmeasure)

print("ROUGE-1:", np.mean(r1))
print("ROUGE-2:", np.mean(r2))
print("ROUGE-L:", np.mean(rl))

ROUGE-1: 0.3509089639209597
ROUGE-2: 0.14702143279231536
ROUGE-L: 0.2511695654765853
